# Sales Forecasting — Exploratory Data Analysis
**Phase 1 | MLOps Project**

This notebook covers:
1. Dataset overview and statistics
2. Sales distributions
3. Temporal patterns (daily, weekly, monthly, yearly)
4. Store-level analysis
5. Promotion and holiday effects
6. Feature correlation analysis
7. Key insights summary

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Styling
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})

RAW_PATH = '../data/raw/sales_data.csv'
PROCESSED_TRAIN = '../data/processed/train.csv'

## 1. Dataset Overview

In [ ]:
df = pd.read_csv(RAW_PATH, parse_dates=['date'])
df_open = df[df['open'] == 1].copy()

print('=== Dataset Summary ===')
print(f'Total rows      : {len(df):,}')
print(f'Open-store rows : {len(df_open):,}')
print(f'Date range      : {df.date.min().date()} → {df.date.max().date()}')
print(f'Stores          : {df.store_id.nunique()}')
print(f'Columns         : {list(df.columns)}')
print()
df.head(10)

In [ ]:
print('=== Descriptive Statistics (open stores) ===')
df_open[['sales','customers','promo','school_holiday']].describe().round(2)

In [ ]:
print('=== Missing Values ===')
missing = df.isnull().sum()
print(missing[missing > 0] if missing.sum() > 0 else 'No missing values')

## 2. Sales Distribution

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Raw distribution
axes[0].hist(df_open['sales'], bins=60, color='steelblue', edgecolor='white', alpha=0.8)
axes[0].set_title('Sales Distribution (raw)')
axes[0].set_xlabel('Daily Sales')
axes[0].set_ylabel('Count')

# Log distribution
axes[1].hist(np.log1p(df_open['sales']), bins=60, color='coral', edgecolor='white', alpha=0.8)
axes[1].set_title('Log(Sales + 1) Distribution')
axes[1].set_xlabel('log(Sales + 1)')

# Boxplot per store
df_open.boxplot(column='sales', by='store_id', ax=axes[2], 
                boxprops=dict(color='steelblue'),
                medianprops=dict(color='red', linewidth=2))
axes[2].set_title('Sales Distribution by Store')
axes[2].set_xlabel('Store ID')
axes[2].set_ylabel('Sales')
plt.suptitle('')

plt.tight_layout()
plt.savefig('../data/processed/eda_sales_distribution.png', bbox_inches='tight')
plt.show()
print('Skewness:', round(df_open.sales.skew(), 3))

## 3. Temporal Patterns

In [ ]:
# Daily aggregate sales across all stores
daily = df_open.groupby('date')['sales'].sum().reset_index()

fig, axes = plt.subplots(3, 1, figsize=(16, 12))

# Full time series
axes[0].plot(daily['date'], daily['sales'], color='steelblue', linewidth=0.8, alpha=0.8)
axes[0].fill_between(daily['date'], daily['sales'], alpha=0.15, color='steelblue')
axes[0].set_title('Total Daily Sales — All Stores')
axes[0].set_ylabel('Total Sales')
axes[0].xaxis.set_major_locator(mdates.MonthLocator(interval=3))
axes[0].xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))

# Monthly average
monthly = df_open.groupby(df_open['date'].dt.to_period('M'))['sales'].mean()
monthly.index = monthly.index.to_timestamp()
axes[1].bar(monthly.index, monthly.values, width=25, color='teal', alpha=0.8, edgecolor='white')
axes[1].set_title('Monthly Average Sales')
axes[1].set_ylabel('Avg Sales per Store-Day')
axes[1].xaxis.set_major_locator(mdates.MonthLocator(interval=3))
axes[1].xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))

# Day-of-week pattern
dow_labels = ['Mon','Tue','Wed','Thu','Fri','Sat','Sun']
dow_avg = df_open.groupby(df_open['date'].dt.dayofweek)['sales'].mean()
axes[2].bar(dow_labels[:len(dow_avg)], dow_avg.values, color='coral', alpha=0.85, edgecolor='white')
axes[2].set_title('Average Sales by Day of Week')
axes[2].set_ylabel('Avg Sales')

plt.tight_layout()
plt.savefig('../data/processed/eda_temporal_patterns.png', bbox_inches='tight')
plt.show()

In [ ]:
# Monthly seasonality heatmap
df_open['year'] = df_open['date'].dt.year
df_open['month'] = df_open['date'].dt.month

monthly_heat = df_open.groupby(['year','month'])['sales'].mean().unstack()
monthly_heat.columns = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']

fig, ax = plt.subplots(figsize=(14, 4))
sns.heatmap(monthly_heat, annot=True, fmt='.0f', cmap='YlOrRd', ax=ax, linewidths=0.5)
ax.set_title('Average Daily Sales Heatmap (Year × Month)')
plt.tight_layout()
plt.savefig('../data/processed/eda_seasonality_heatmap.png', bbox_inches='tight')
plt.show()

## 4. Store-Level Analysis

In [ ]:
store_summary = df_open.groupby('store_id').agg(
    avg_sales=('sales', 'mean'),
    total_sales=('sales', 'sum'),
    std_sales=('sales', 'std'),
    promo_rate=('promo', 'mean'),
    store_type=('store_type', 'first'),
    assortment=('assortment', 'first'),
).reset_index()

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Avg sales per store
axes[0].barh(store_summary['store_id'].astype(str), store_summary['avg_sales'],
             color='steelblue', alpha=0.85)
axes[0].set_title('Average Daily Sales per Store')
axes[0].set_xlabel('Avg Sales')
axes[0].set_ylabel('Store ID')

# Sales by store type
type_avg = df_open.groupby('store_type')['sales'].mean()
axes[1].bar(type_avg.index, type_avg.values, color=['steelblue','coral','teal','gold'], alpha=0.85)
axes[1].set_title('Average Sales by Store Type')
axes[1].set_xlabel('Store Type')
axes[1].set_ylabel('Avg Sales')

# Sales vs competition distance
axes[2].scatter(df_open['competition_distance'], df_open['sales'],
                alpha=0.3, s=5, color='purple')
axes[2].set_title('Sales vs Competition Distance')
axes[2].set_xlabel('Competition Distance (m)')
axes[2].set_ylabel('Sales')

plt.tight_layout()
plt.savefig('../data/processed/eda_store_analysis.png', bbox_inches='tight')
plt.show()
print(store_summary[['store_id','avg_sales','promo_rate','store_type','assortment']].to_string(index=False))

## 5. Promotion & Holiday Effects

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Promo effect
promo_avg = df_open.groupby('promo')['sales'].mean()
axes[0].bar(['No Promo','Promo'], promo_avg.values, color=['steelblue','coral'], alpha=0.85)
axes[0].set_title('Avg Sales: Promo vs No Promo')
axes[0].set_ylabel('Average Sales')
for i, v in enumerate(promo_avg.values):
    axes[0].text(i, v + 50, f'{v:.0f}', ha='center', fontsize=10)

# School holiday effect
sh_avg = df_open.groupby('school_holiday')['sales'].mean()
axes[1].bar(['No Holiday','School Holiday'], sh_avg.values, color=['teal','gold'], alpha=0.85)
axes[1].set_title('Avg Sales: School Holiday Effect')
axes[1].set_ylabel('Average Sales')

# Promo lift by store type
promo_type = df_open.groupby(['store_type','promo'])['sales'].mean().unstack()
promo_type.columns = ['No Promo','Promo']
promo_type['lift_pct'] = ((promo_type['Promo'] - promo_type['No Promo']) / promo_type['No Promo'] * 100).round(1)
axes[2].bar(promo_type.index, promo_type['lift_pct'], color='coral', alpha=0.85)
axes[2].set_title('Promo Lift % by Store Type')
axes[2].set_xlabel('Store Type')
axes[2].set_ylabel('Lift (%)')
axes[2].axhline(0, color='black', linewidth=0.8)

plt.tight_layout()
plt.savefig('../data/processed/eda_promo_holiday.png', bbox_inches='tight')
plt.show()
print('Promo lift overall:', round((promo_avg[1] - promo_avg[0]) / promo_avg[0] * 100, 1), '%')

## 6. Feature Correlation

In [ ]:
train = pd.read_csv(PROCESSED_TRAIN, parse_dates=['date'])

numeric_cols = train.select_dtypes(include=[np.number]).columns.tolist()
corr_with_target = train[numeric_cols].corr()['sales'].drop('sales').sort_values(key=abs, ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Top correlations with sales
top_corr = corr_with_target.head(15)
colors = ['coral' if v > 0 else 'steelblue' for v in top_corr.values]
axes[0].barh(top_corr.index[::-1], top_corr.values[::-1], color=colors[::-1], alpha=0.85)
axes[0].axvline(0, color='black', linewidth=0.8)
axes[0].set_title('Top 15 Feature Correlations with Sales')
axes[0].set_xlabel('Pearson Correlation')

# Heatmap of top features
top_features = corr_with_target.head(10).index.tolist() + ['sales']
corr_matrix = train[top_features].corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, ax=axes[1], linewidths=0.5, annot_kws={'size': 8})
axes[1].set_title('Correlation Heatmap (Top Features)')
plt.tight_layout()
plt.savefig('../data/processed/eda_correlations.png', bbox_inches='tight')
plt.show()

print('Top 10 features by correlation with sales:')
print(corr_with_target.head(10).to_string())

## 7. Key Insights Summary

In [ ]:
promo_lift = round((df_open[df_open.promo==1]['sales'].mean() - df_open[df_open.promo==0]['sales'].mean())
                   / df_open[df_open.promo==0]['sales'].mean() * 100, 1)
best_store = store_summary.loc[store_summary['avg_sales'].idxmax(), 'store_id']
best_dow = dow_avg.idxmax()
dow_names = {0:'Monday',1:'Tuesday',2:'Wednesday',3:'Thursday',4:'Friday',5:'Saturday',6:'Sunday'}

insights = {
    'Total training rows (after feature engineering)': f"{len(train):,}",
    'Feature count (excluding target)': f"{len(numeric_cols) - 1}",
    'Overall promotion lift': f"{promo_lift}%",
    'Best performing day of week': dow_names.get(best_dow, best_dow),
    'Best performing store': f"Store {best_store}",
    'Peak month': 'December (holiday season)',
    'Sales distribution': 'Right-skewed → log-transform recommended for modelling',
    'Strongest predictor': corr_with_target.index[0],
    'Data leakage risk': 'None — time-based split used',
}

print('=' * 52)
print('           KEY EDA INSIGHTS')
print('=' * 52)
for k, v in insights.items():
    print(f'  {k:<42} {v}')
print('=' * 52)
print('\nPhase 1 complete. Proceed to Phase 2: Model Training.')